# Taller: Arquitectura Big Data para IoT (Ingesta Masiva + Batch y Streaming)

**Objetivo:** Diseñar y simular una arquitectura empresarial para ingestión masiva de IoT (millones de eventos/min) con análisis en streaming y batch sobre Lakehouse.

---
## 1. Caso

- Empresa industrial con 50k dispositivos IoT en 12 plantas.
- Requisitos: telemetría en tiempo real para alertas (P95<2s) y reporting diario (SLAs 8am).
- Datos: sensores (temperatura, vibración, presión), logs, imágenes ocasionales.

---
## 2. Arquitectura Propuesta (ASCII)

```
Devices (MQTT) → Gateway Edge → Kafka Topics (iot_raw, iot_alerts)
                           │
                           ├─► Stream Proc (Flink/Spark): ventanas, joins, reglas
                           │          │
                           │          ├─► Online Store / Alerts API (low-latency)
                           │          └─► Feature Store Online (para ML en tiempo real)
                           │
                           └─► Ingesta a Lake (Bronze) → ETL a Silver → Gold (BI/ML)
                                      │                   │
                                      ├─► Batch Proc (Spark + dbt)
                                      └─► Data Products / Dashboards / Warehouse
```

Controles transversales: catálogo+linaje, calidad, gobernanza, observabilidad, costos (FinOps).

---
## 3. Simulador: Eventos IoT y Ventanas Temporales

In [ ]:
from dataclasses import dataclass
from typing import List, Dict
from datetime import datetime, timedelta
from collections import deque, defaultdict
import random

random.seed(7)

@dataclass
class IoTEvent:
    device_id: str
    ts: datetime
    temperature: float
    vibration: float
    pressure: float

class WindowProcessor:
    def __init__(self, window_seconds: int = 60):
        self.window = timedelta(seconds=window_seconds)
        self.buffers: Dict[str, deque] = defaultdict(deque)

    def ingest(self, event: IoTEvent) -> Dict:
        buf = self.buffers[event.device_id]
        buf.append(event)
        # Remove old
        min_ts = event.ts - self.window
        while buf and buf[0].ts < min_ts:
            buf.popleft()
        # Compute aggregates
        temps = [e.temperature for e in buf]
        vibs = [e.vibration for e in buf]
        pres = [e.pressure for e in buf]
        return {
            'device_id': event.device_id,
            'window_size': len(buf),
            'temp_avg': sum(temps)/len(temps),
            'vib_p95': sorted(vibs)[int(0.95*len(vibs))-1],
            'pressure_max': max(pres)
        }

# Simular 3 dispositivos por 120s
proc = WindowProcessor(window_seconds=60)
now = datetime.now()
devices = ['dev_001','dev_002','dev_003']
alerts = []

for t in range(120):
    for d in devices:
        e = IoTEvent(
            device_id=d,
            ts=now + timedelta(seconds=t),
            temperature= random.uniform(60, 100),
            vibration= random.uniform(0.2, 1.2),
            pressure= random.uniform(80, 130)
        )
        agg = proc.ingest(e)
        # Regla simple: alerta si vib_p95 alto y temp_avg > 90
        if agg['vib_p95'] > 1.0 and agg['temp_avg'] > 90:
            alerts.append((e.ts, d, 'ALERT_HIGH_RISK'))

print(f'Total alerts: {len(alerts)}')
print('Sample alerts:', alerts[:5])

## 4. Job Batch Diario (Silver → Gold)

Pasos típicos (Spark + dbt):
1. Compaction y clustering de archivos.
2. Enriquecimiento con catálogos (plantas, líneas, mantenimientos).
3. Cálculo de KPIs: OEE, MTBF, MTTR, tasas de alertas por línea.
4. Validación de calidad (completitud, outliers) y publish a Gold.

## 5. Observabilidad y Costos (SLOs/FinOps)

- SLO streaming: P95<2s, error rate <0.1%, lag <10s.
- SLO batch: job <45min, antes de 8am.
- Costos: egress evitado, compaction programado, auto-scaling, right-sizing.

## 6. Actividades del Taller

1. Diseña el esquema de tópicos Kafka (keys/particiones/retención).
2. Define el plan de particionamiento en Lake (por planta/fecha).
3. Propón reglas de calidad y alertas (operacionales y de datos).
4. Esboza un dashboard mínimo con métricas clave (streaming y batch).
5. Estima costos mensuales (storage, compute, streaming) y optimizaciones.

---
**Referencias:** Flink 1.18 docs, Spark 3.5, Kafka Design Guide, Delta/Iceberg/Hudi, Data Observability guides.